In [1]:
import pandas as pd

In [24]:
fields = pd.read_csv("./data/2026/hakukohteiden koulutusalat.csv", delimiter=";")

In [13]:
programmes = pd.read_csv("./analysis/2026/study_programmes.csv", delimiter=";")

In [26]:
matches = pd.merge(programmes, fields[["id", "study_field"]], on="id", how="left")

In [31]:
unmatched_programmes = matches[matches["study_field"].isnull()]

In [34]:
matches.to_csv("./analysis/2026/study_programmes_with_fields.csv", index=False, sep=";")

In [63]:
programmes_old = pd.read_csv("./analysis/2025/study_programmes.csv", delimiter=";")

In [44]:
from rapidfuzz import process, fuzz

In [65]:
matches["match_key"] = (
    matches["name"].fillna("") + " | " +
    matches["university"].fillna("")
)

In [87]:
matches["study_field"].unique()

array(['tekniikka', 'fysiikka', 'kemia', 'matemaattiset tieteet',
       'opettajankoulutukset', 'tietojenkäsittelytieteet', 'nanotiede',
       'biokemia ja molekyylibiotieteet', 'eläinlääketiede', 'farmasia',
       'hammaslääketiede', 'lääketiede', 'biolääketiede',
       'ravitsemustiede', 'biologia ja ympäristötieteet',
       'elintarviketieteet', 'maatalous- ja metsätieteet', 'geotieteet',
       'maantiede', 'logopedia', 'psykologia', 'terveys- ja hoitotieteet',
       'liikuntatieteet', 'kasvatustieteet', 'liikuntapedagogiikka',
       'kauppatiede', 'taloustiede', 'tietojärjestelmätiede',
       'oikeustieteet', 'yhteiskuntatieteet', 'sosiaalitieteet',
       'viestintätieteet', 'hallintotieteet', 'filosofia', 'historia',
       'kulttuurien ja taiteiden tutkimus', 'teologia',
       'vieraat kielet ja kielitieteet', 'kotimaiset kielet',
       'kirjallisuus'], dtype=object)

In [67]:
programmes_old["match_key"] = (
    programmes_old["name"].fillna("") + " | " +
    programmes_old["university"].fillna("")
)

In [68]:
choices = matches["match_key"].tolist()

In [73]:
def best_match(key):
    result = process.extractOne(key, choices, scorer=fuzz.WRatio)
    if result:
        match, score, _ = result
        return pd.Series([match, score])
    return pd.Series([None, None])

In [74]:
programmes_old[["matched_key", "match_score"]] = programmes_old["match_key"].apply(best_match)

In [84]:
old_matches = programmes_old.merge(
    matches[["match_key", "study_field"]],
    left_on="matched_key",
    right_on="match_key",
    how="left"
)

In [85]:
old_matches

,id,name,university,faculty,exam,match_key_x,matched_key,match_score,match_key_y,study_field
0,1.2.246.562.20.00000000000000056120,"Energia- ja konetekniikka, tekniikan kandidaat...",Aalto-yliopisto,Insinööritieteiden korkeakoulu,A,"Energia- ja konetekniikka, tekniikan kandidaat...","Energia- ja konetekniikka, tekniikan kandidaat...",93.775934,"Energia- ja konetekniikka, tekniikan kandidaat...",tekniikka
1,1.2.246.562.20.00000000000000056121,"Kestävät yhdyskunnat, tekniikan kandidaatti ja...",Aalto-yliopisto,Insinööritieteiden korkeakoulu,A,"Kestävät yhdyskunnat, tekniikan kandidaatti ja...","Kestävät yhdyskunnat, tekniikan kandidaatti ja...",93.506494,"Kestävät yhdyskunnat, tekniikan kandidaatti ja...",tekniikka
2,1.2.246.562.20.00000000000000056123,"Kiinteistötalous ja geoinformatiikka, tekniika...",Aalto-yliopisto,Insinööritieteiden korkeakoulu,A,"Kiinteistötalous ja geoinformatiikka, tekniika...","Kiinteistötalous ja geoinformatiikka, tekniika...",94.296578,"Kiinteistötalous ja geoinformatiikka, tekniika...",tekniikka
3,1.2.246.562.20.00000000000000056124,"Rakennustekniikka, tekniikan kandidaatti ja di...",Aalto-yliopisto,Insinööritieteiden korkeakoulu,A,"Rakennustekniikka, tekniikan kandidaatti ja di...","Rakennustekniikka, tekniikan kandidaatti ja di...",93.333333,"Rakennustekniikka, tekniikan kandidaatti ja di...",tekniikka
4,1.2.246.562.20.00000000000000056112,"Kemian tekniikka, tekniikan kandidaatti ja dip...",Aalto-yliopisto,Kemian tekniikan korkeakoulu,A,"Kemian tekniikka, tekniikan kandidaatti ja dip...","Kemian tekniikka, tekniikan kandidaatti ja dip...",93.273543,"Kemian tekniikka, tekniikan kandidaatti ja dip...",tekniikka
...,...,...,...,...,...,...,...,...,...,...
381,1.2.246.562.20.00000000000000056135,"Finska språket, utbildningsprogrammet i språk,...",Åbo Akademi,Fakulteten för humaniora,I,"Finska språket, utbildningsprogrammet i språk,...","Finska språket, utbildningsprogrammet i språk,...",100.000000,"Finska språket, utbildningsprogrammet i språk,...",kotimaiset kielet
382,1.2.246.562.20.00000000000000056136,"Franska språket och litteraturen, utbildningsp...",Åbo Akademi,Fakulteten för humaniora,I,"Franska språket och litteraturen, utbildningsp...","Franska språket och litteraturen, utbildningsp...",100.000000,"Franska språket och litteraturen, utbildningsp...",vieraat kielet ja kielitieteet
383,1.2.246.562.20.00000000000000056140,"Ryska språket och litteraturen, utbildningspro...",Åbo Akademi,Fakulteten för humaniora,I,"Ryska språket och litteraturen, utbildningspro...","Ryska språket och litteraturen, utbildningspro...",100.000000,"Ryska språket och litteraturen, utbildningspro...",vieraat kielet ja kielitieteet
384,1.2.246.562.20.00000000000000056139,"Svenska språket, utbildningsprogrammet i språk...",Åbo Akademi,Fakulteten för humaniora,I,"Svenska språket, utbildningsprogrammet i språk...","Svenska språket, utbildningsprogrammet i språk...",100.000000,"Svenska språket, utbildningsprogrammet i språk...",kotimaiset kielet


In [86]:
old_matches.to_csv("./analysis/2025/study_programmes_with_fields.csv", index=False, sep=";")

In [88]:
import json

In [ ]:
with open("./analysis/2025/study_programmes.json", encoding="utf-8") as f:
    study_programmes = json.load(f)


In [105]:
study_fields = pd.read_csv("./analysis/2025/study_programmes_with_fields.csv", delimiter=";")

In [106]:
for programme_id in study_programmes.keys():
    programme = study_programmes[programme_id]
    
    matching_row = study_fields[study_fields["id"] == programme_id]
    if not matching_row.empty:
        study_field = matching_row.iloc[0]["study_field"]
        programme["study_field"] = study_field
    else:
        print(f"No matching row found for programme ID: {programme_id}")

In [107]:
with open("./analysis/2025/study_programmes.json", "w") as f:
    json.dump(study_programmes, f, indent=4, ensure_ascii=False)